# Análise Comparativa: H-CAT vs PyHard

Este notebook compara as ferramentas **H-CAT** e **PyHard** no objetivo central do TCC:
**identificar amostras rotuladas incorretamente (hard samples) em datasets tabulares**.

## Dados de entrada

A análise parte de datasets **meta já calculados**:

| Arquivo | Amostras | Medidas | Coluna alvo |
|---------|----------|---------|--------------|
| `datasets/diabetes_meta_pyhard_10_20_30pct.csv` | 162 826 | 11 | `problema_qualidade` |
| `datasets/diabetes_meta_hcat_10_20_30pct.csv` | 162 826 | 10 | `problema_qualidade` |
| `datasets/covertype_meta_pyhard_10_20_30pct.csv` | 192 000 | 13 (11 válidas) | `problema_qualidade` |
| `datasets/covertype_meta_hcat_10_20_30pct.csv` | 192 000 | 11 | `problema_qualidade` |

Cada CSV contém, por instância, o valor de cada medida de hardness e a coluna `problema_qualidade` que indica se a amostra foi ou não perturbada (`'sim'` / `'nao'`), atuando como *ground-truth*.

## Protocolo experimental

- **Perturbação**: *uniform mislabeling* (troca aleatória da classe) em três proporções consolidadas: 10%, 20% e 30%.
- **Avaliação**: para cada medida, calcula-se o AUC-ROC e o AUC-PRC na tarefa binária **detectar `problema_qualidade='sim'`**, escolhendo automaticamente a direção (direta ou invertida) que maximiza o AUC-ROC.
- **Dataset agregado**: as três proporções estão unificadas no mesmo arquivo; a avaliação mede portanto a capacidade global de detecção, não decompõe por proporção individual.

## Análises produzidas

1. **AUC-ROC / AUC-PRC por medida** (tabela comparativa)
2. **Heatmap de AUC-ROC** H-CAT × PyHard lado a lado nos dois datasets
3. **Distribuição das medidas** em amostras perturbadas vs. limpas
4. **Correlação entre medidas** heatmap inter-ferramentas
5. **Concordância de ranking** H-CAT × PyHard por instância
6. **Exemplos de instâncias discordantes** — casos onde uma ferramenta detecta e a outra não

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import average_precision_score, roc_auc_score

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150

BASE_DIR = Path.cwd().resolve()
while BASE_DIR.name and not (BASE_DIR / "datasets").is_dir():
    if BASE_DIR.parent == BASE_DIR:
        break
    BASE_DIR = BASE_DIR.parent

META_DIR = BASE_DIR / "datasets"
OUTPUT_DIR = BASE_DIR / "data" / "comparison_results"
FIGURES_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

META_FILES = {
    ("diabetes", "pyhard"): META_DIR / "diabetes_meta_pyhard_10_20_30pct.csv",
    ("diabetes", "hcat"):   META_DIR / "diabetes_meta_hcat_10_20_30pct.csv",
    ("covertype", "pyhard"): META_DIR / "covertype_meta_pyhard_10_20_30pct.csv",
    ("covertype", "hcat"):   META_DIR / "covertype_meta_hcat_10_20_30pct.csv",
}


## 2. Carregamento dos datasets meta

Carrega os quatro CSVs, identifica colunas ID (`encounter_id` para Diabetes, `instance_id` para Covertype) e extrai a coluna de target `problema_qualidade` como vetor binário `y_true`.

Para cada medida, reporta:
- contagem de valores válidos,
- quantos NaN (preenchidos com a mediana da própria medida),
- quantas instâncias constantes (descartadas).

In [ ]:
ID_COL_BY_DATASET = {"diabetes": "encounter_id", "covertype": "instance_id"}


def load_meta(path: Path, dataset: str):
    df = pd.read_csv(path)
    id_col = ID_COL_BY_DATASET[dataset]
    assert id_col in df.columns and "problema_qualidade" in df.columns
    y_true = (df["problema_qualidade"] == "sim").astype(int)
    measures = [c for c in df.columns if c not in {id_col, "problema_qualidade"}]
    return df, y_true, measures


meta_store: dict[tuple[str, str], dict] = {}
for (dataset, tool), path in META_FILES.items():
    df, y_true, measures = load_meta(path, dataset)
    const_cols = [m for m in measures if df[m].nunique(dropna=True) <= 1]
    meta_store[(dataset, tool)] = dict(
        df=df, y_true=y_true, measures=measures, const_cols=const_cols,
    )


## 3. Cálculo de AUC-ROC e AUC-PRC por medida

Para cada medida:

1. Imputar NaN com a mediana da própria medida (robusto a outliers).
2. Testar AUC na direção **direta** (valor alto ⇒ mais provavelmente perturbado) e **invertida**.
3. Reportar a direção que maximiza AUC-ROC e também AUC-PRC para essa mesma direção.
4. Marcar como degenerado quando AUC-ROC ≈ 0.5.

In [ ]:
def compute_auc_for_measure(scores: np.ndarray, y_true: np.ndarray) -> dict:
    scores = np.asarray(scores, dtype=float)
    y_true = np.asarray(y_true, dtype=int)
    n_nan = int(np.isnan(scores).sum())
    if n_nan > 0:
        median = np.nanmedian(scores)
        scores = np.where(np.isnan(scores), median, scores)
    if np.unique(scores).size <= 1:
        return dict(
            auc_roc=np.nan, auc_prc=np.nan, direction="n/a",
            n_nan=n_nan, degenerate=True, note="constante",
        )
    try:
        roc_dir = roc_auc_score(y_true, scores)
        roc_inv = roc_auc_score(y_true, -scores)
    except ValueError as err:
        return dict(
            auc_roc=np.nan, auc_prc=np.nan, direction="n/a",
            n_nan=n_nan, degenerate=True, note=f"erro: {err}",
        )
    if roc_dir >= roc_inv:
        final_roc = roc_dir
        final_prc = average_precision_score(y_true, scores)
        direction = "direta"
    else:
        final_roc = roc_inv
        final_prc = average_precision_score(y_true, -scores)
        direction = "invertida"
    degenerate = abs(final_roc - 0.5) < 1e-3
    return dict(
        auc_roc=float(final_roc),
        auc_prc=float(final_prc),
        direction=direction,
        n_nan=n_nan,
        degenerate=bool(degenerate),
        note="",
    )

records: list[dict] = []
for (dataset, tool), store in meta_store.items():
    df = store["df"]
    y_true = store["y_true"].values
    for measure in store["measures"]:
        if measure in store["const_cols"]:
            records.append(dict(
                dataset=dataset, tool=tool, measure=measure,
                auc_roc=np.nan, auc_prc=np.nan, direction="n/a",
                n_nan=int(df[measure].isna().sum()), degenerate=True, note="constante",
            ))
            continue
        result = compute_auc_for_measure(df[measure].values, y_true)
        records.append(dict(dataset=dataset, tool=tool, measure=measure, **result))

auc_df = pd.DataFrame(records)
auc_df = auc_df.sort_values(["dataset", "tool", "auc_roc"], ascending=[True, True, False]).reset_index(drop=True)
auc_df

In [ ]:
summary = (
    auc_df.dropna(subset=["auc_roc"])
    .groupby(["dataset", "tool"])
    .agg(
        n_medidas_validas=("measure", "count"),
        auc_roc_mediana=("auc_roc", "median"),
        auc_roc_media=("auc_roc", "mean"),
        auc_roc_max=("auc_roc", "max"),
        auc_roc_min=("auc_roc", "min"),
        auc_prc_mediana=("auc_prc", "median"),
        n_degeneradas=("degenerate", "sum"),
    )
    .round(4)
)
auc_df.to_csv(OUTPUT_DIR / "comparison_summary_auc.csv", index=False)
summary


## 4. Heatmap comparativo de AUC-ROC

Visualiza de forma compacta o desempenho de cada medida em cada (dataset, ferramenta).

In [ ]:
def plot_auc_barplot(auc_df: pd.DataFrame, dataset: str, metric: str = "auc_roc", ax=None):
    sub = auc_df.query("dataset == @dataset").copy()
    sub["tool_label"] = sub["tool"].str.upper()
    sub = sub.sort_values(["tool", metric], ascending=[True, False])
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(sub) + 1)))
    palette = {"HCAT": "#2E86AB", "PYHARD": "#A23B72"}
    sns.barplot(
        data=sub, y="measure", x=metric, hue="tool_label",
        palette=palette, ax=ax, orient="h",
    )
    ax.axvline(0.5, color="black", linestyle="--", linewidth=0.8, alpha=0.6, label="AUC=0.5 (aleatório)")
    ax.set_xlabel(metric.upper().replace("_", "-"))
    ax.set_ylabel("Medida")
    ax.set_title(f"{dataset.title()} — {metric.upper().replace('_', '-')} por medida")
    ax.set_xlim(0.35, 1.0)
    ax.legend(loc="lower right")
    return ax

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
plot_auc_barplot(auc_df, "diabetes", "auc_roc", ax=axes[0])
plot_auc_barplot(auc_df, "covertype", "auc_roc", ax=axes[1])
plt.tight_layout()
fig.savefig(FIGURES_DIR / "auc_roc_barplot.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
plot_auc_barplot(auc_df, "diabetes", "auc_prc", ax=axes[0])
plot_auc_barplot(auc_df, "covertype", "auc_prc", ax=axes[1])
plt.tight_layout()
fig.savefig(FIGURES_DIR / "auc_prc_barplot.png", bbox_inches="tight")
plt.show()

## 5. Distribuição das medidas em amostras perturbadas vs. limpas

Violinplots mostram a distribuição completa da medida separada por `problema_qualidade`. Uma medida útil exibe separação clara entre as duas distribuições.

Para reduzir ruído visual, as medidas são normalizadas para [0,1] e orientadas na direção ótima (encontrada no passo 3).

In [ ]:
def build_long_df(dataset: str, tool: str, auc_df: pd.DataFrame, meta_store: dict) -> pd.DataFrame:
    store = meta_store[(dataset, tool)]
    df_meta = store["df"]
    y_true = store["y_true"]
    rows = []
    directions = auc_df.query("dataset == @dataset and tool == @tool").set_index("measure")["direction"]
    for measure in store["measures"]:
        if measure in store["const_cols"]:
            continue
        raw = df_meta[measure].values.astype(float)
        if np.isnan(raw).any():
            raw = np.where(np.isnan(raw), np.nanmedian(raw), raw)
        mn, mx = raw.min(), raw.max()
        norm = (raw - mn) / (mx - mn + 1e-12)
        direction = directions.get(measure, "direta")
        if direction == "invertida":
            norm = 1 - norm
        rows.append(pd.DataFrame({
            "measure": measure,
            "score_norm": norm,
            "perturbado": np.where(y_true == 1, "sim", "nao"),
        }))
    return pd.concat(rows, ignore_index=True)

def plot_violin(dataset: str, tool: str, auc_df: pd.DataFrame, meta_store: dict, ax=None):
    long = build_long_df(dataset, tool, auc_df, meta_store)
    order = (
        auc_df.query("dataset == @dataset and tool == @tool")
        .dropna(subset=["auc_roc"])
        .sort_values("auc_roc", ascending=False)["measure"]
        .tolist()
    )
    long["measure"] = pd.Categorical(long["measure"], categories=order, ordered=True)
    if ax is None:
        fig, ax = plt.subplots(figsize=(max(8, 0.9 * len(order)), 5))
    sns.violinplot(
        data=long, x="measure", y="score_norm", hue="perturbado",
        split=True, inner="quartile", palette={"nao": "#9DBEBB", "sim": "#E76F51"}, ax=ax,
    )
    ax.set_title(f"{dataset.title()} — {tool.upper()} (direção ótima aplicada)")
    ax.set_ylabel("medida (normalizada)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
    return ax

for dataset in ("diabetes", "covertype"):
    fig, axes = plt.subplots(2, 1, figsize=(max(10, 0.9 * 11), 10))
    plot_violin(dataset, "hcat", auc_df, meta_store, ax=axes[0])
    plot_violin(dataset, "pyhard", auc_df, meta_store, ax=axes[1])
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / f"violin_{dataset}.png", bbox_inches="tight")
    plt.show()

## 9. Síntese final e artefatos salvos

Consolidação do que foi produzido e dos insights principais para levar ao texto do TCC.

In [ ]:
for dataset in ("diabetes", "covertype"):
    sub = auc_df.query("dataset == @dataset and not degenerate").dropna(subset=["auc_roc"])
    print(f"\n=== {dataset.upper()} — top 3 medidas por AUC-ROC ===")
    for tool in ("hcat", "pyhard"):
        top = sub.query("tool == @tool").sort_values("auc_roc", ascending=False).head(3)
        if top.empty:
            continue
        print(f"  {tool.upper()}:")
        for _, row in top.iterrows():
            print(f"    {row['measure']:<18} AUC-ROC={row['auc_roc']:.4f}  AUC-PRC={row['auc_prc']:.4f}  ({row['direction']})")
